# BTP step S1b on Colab: TL-HF equivalence checks (plan Table 5)

Runs `scripts/s1_equivalence.sh` per model and writes results to Google Drive.

**Before running**
1. Runtime > Change runtime type > GPU. Both the HF and the TransformerLens copy sit on the GPU at once:
   gemma-2-2b-it (float32) needs an **L4 (24 GB) or larger**; gemma-2-9b-it and qwen2.5-7b-instruct (bf16) need an **A100 / H100**. A free T4 (15 GB) fits none of them.
2. Put `btp_code.zip` in `MyDrive/BTP/`.
3. Gemma is gated: accept the licence on its HuggingFace page, then add a Colab secret (key icon, left bar) named `HF_TOKEN` with notebook access on.

S1c (`s1_sae.sh`) is deliberately not here: `prereg/E0.md` must be reviewed and committed first.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/btp && mkdir -p /content/btp && unzip -q /content/drive/MyDrive/BTP/btp_code.zip -d /content/btp && ls /content/btp

In [ ]:
# requirements.txt needs Python 3.13 (TransformerLens 2.x caps numpy<2 on <=3.12), so build a 3.13 venv with uv.
!pip -q install uv
!uv venv --python 3.13 /content/venv
!uv pip install --python /content/venv/bin/python -r /content/btp/requirements.txt
!/content/venv/bin/python -c "import torch, transformer_lens, numpy; print(torch.__version__, torch.cuda.is_available(), transformer_lens.__version__ if hasattr(transformer_lens,'__version__') else '', numpy.__version__)"

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # read from Colab secrets; never printed
os.environ['PATH'] = '/content/venv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/content/venv'

In [ ]:
# Pick the models that fit this GPU. One at a time; each writes reports/m0/equivalence_<model>.json
MODELS = ['gemma-2-2b-it']          # add 'gemma-2-9b-it', 'qwen2.5-7b-instruct' on an A100/H100
for m in MODELS:
    !cd /content/btp && bash scripts/s1_equivalence.sh {m} cuda
    !cat /content/btp/reports/m0/equivalence_{m}.json

In [ ]:
# Copy results back to Drive
!mkdir -p /content/drive/MyDrive/BTP/reports && cp -r /content/btp/reports/m0 /content/drive/MyDrive/BTP/reports/ && ls -la /content/drive/MyDrive/BTP/reports/m0